# Descompresión y Análisis de Estructura de Datasets Raw

Este notebook tiene el objetivo iterar por todos los archivos `.zip` en la carpeta `Raw` del proyecto. Por cada archivo:
1. Lo copia de Google Drive al entorno local (`/content/`).
2. Mide el **tiempo de descompresión**.
3. **Borra el archivo `.zip` local** inmediatamente para liberar espacio en Colab.
4. Analiza la **estructura de carpetas**, **formatos de archivo** (`.png`, `.dcm`, `.nii`, etc.) y **tamaño total** ocupado en disco.


In [1]:
from google.colab import drive
import os
import time
import shutil
from pathlib import Path
from collections import Counter

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def format_size(size_bytes):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0

def analyze_directory(dir_path):
    total_size = 0
    file_count = 0
    folder_count = 0
    extensions = Counter()

    for root, dirs, files in os.walk(dir_path):
        folder_count += len(dirs)
        file_count += len(files)
        for f in files:
            filepath = os.path.join(root, f)
            if not os.path.islink(filepath):
                total_size += os.path.getsize(filepath)

            ext = os.path.splitext(f)[1].lower()
            if ext == '':
                ext = 'sin extension'
            extensions[ext] += 1

    return total_size, file_count, folder_count, dict(extensions)

In [3]:
RAW_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/'
LOCAL_DEST = '/content/datasets/'

os.makedirs(LOCAL_DEST, exist_ok=True)

print("Buscando datasets en Drive...")
zip_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith('.zip')]
zip_files.sort()

print(f"Encontrados {len(zip_files)} archivos ZIP:")
for z in zip_files:
    size = os.path.getsize(os.path.join(RAW_DIR, z))
    print(f" - {z} ({format_size(size)})")

Buscando datasets en Drive...
Encontrados 5 archivos ZIP:
 - ISIC 2019.zip (9.10 GB)
 - Knee Osteoarthritis Classification.zip (9.83 GB)
 - Luna16 Lung Cancer Dataset.zip (330.40 MB)
 - NIH Chest X ray 14.zip (42.00 GB)
 - Pancreas Cancer.zip (45.95 GB)


## Extracción y Análisis Automático

In [4]:
results = []

for zip_name in zip_files:
    print("="*70)
    print(f" PROCESANDO: {zip_name}")
    print("="*70)

    drive_zip_path = os.path.join(RAW_DIR, zip_name)
    local_zip_path = os.path.join(LOCAL_DEST, zip_name)
    dataset_name = os.path.splitext(zip_name)[0]
    unzip_dir = os.path.join(LOCAL_DEST, dataset_name)

    # 1. Copiar localmente (más rápido para descomprimir)
    print(" 1. Copiando desde Drive al entorno local...")
    start_cp = time.time()
    !cp "{drive_zip_path}" "{local_zip_path}"
    cp_time = time.time() - start_cp

    # 2. Descomprimir
    os.makedirs(unzip_dir, exist_ok=True)
    print(f" 2. Descomprimiendo en {unzip_dir}...")
    start_uz = time.time()
    !unzip -q "{local_zip_path}" -d "{unzip_dir}"
    uz_time = time.time() - start_uz
    print(f"    Descompresión exitosa. Tiempo: {uz_time:.2f} segundos.")

    # 3. Borrar ZIP local
    print(" 3. Borrando ZIP local para liberar VRAM/Disco...")
    os.remove(local_zip_path)

    # 4. Análisis de estructura
    print(" 4. Analizando estructura y formatos de archivo...")
    start_an = time.time()
    total_size, file_count, folder_count, extensions = analyze_directory(unzip_dir)
    str_size = format_size(total_size)
    an_time = time.time() - start_an

    # Guardar resultados para resumen
    results.append({
        'Dataset': dataset_name,
        'Tiempo Copia (s)': round(cp_time, 2),
        'Tiempo Unzip (s)': round(uz_time, 2),
        'Peso Descomprimido': str_size,
        'Total Archivos': file_count,
        'Total Carpetas': folder_count,
        'Extensiones': extensions
    })

    print("\n   RESULTADOS DEL ANÁLISIS:")
    print(f"   - Peso Total Descomprimido: {str_size}")
    print(f"   - Total Archivos: {file_count:,}")
    print(f"   - Total Carpetas: {folder_count:,}")
    print(f"   - Tipos de archivo encontrados:")
    for ext, count in sorted(extensions.items(), key=lambda x: x[1], reverse=True):
        print(f"        {ext}: {count:,} archivos")
    print("\n")

 PROCESANDO: ISIC 2019.zip
 1. Copiando desde Drive al entorno local...
 2. Descomprimiendo en /content/datasets/ISIC 2019...
    Descompresión exitosa. Tiempo: 121.79 segundos.
 3. Borrando ZIP local para liberar VRAM/Disco...
 4. Analizando estructura y formatos de archivo...

   RESULTADOS DEL ANÁLISIS:
   - Peso Total Descomprimido: 9.13 GB
   - Total Archivos: 25,335
   - Total Carpetas: 2
   - Tipos de archivo encontrados:
        .jpg: 25,331 archivos
        .csv: 2 archivos
        .txt: 2 archivos


 PROCESANDO: Knee Osteoarthritis Classification.zip
 1. Copiando desde Drive al entorno local...
 2. Descomprimiendo en /content/datasets/Knee Osteoarthritis Classification...
    Descompresión exitosa. Tiempo: 118.10 segundos.
 3. Borrando ZIP local para liberar VRAM/Disco...
 4. Analizando estructura y formatos de archivo...

   RESULTADOS DEL ANÁLISIS:
   - Peso Total Descomprimido: 9.92 GB
   - Total Archivos: 12,712
   - Total Carpetas: 13
   - Tipos de archivo encontrados:
 

## Resumen General

In [5]:
import pandas as pd

df_resumen = pd.DataFrame(results)
# Quitamos temporalmente la columna del diccionario para imprimir bonito
col_exts = df_resumen.pop('Extensiones')

display(df_resumen)

print("\n" + "*"*50)
print("FORMATOS POR DATASET")
print("*"*50)
for i, row in df_resumen.iterrows():
    print(f"\n{row['Dataset']}:")
    exts = col_exts[i]
    for ext, count in sorted(exts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {ext}: {count:,}")

,Dataset,Tiempo Copia (s),Tiempo Unzip (s),Peso Descomprimido,Total Archivos,Total Carpetas
0,ISIC 2019,191.38,121.79,9.13 GB,25335,2
1,Knee Osteoarthritis Classification,132.87,118.10,9.92 GB,12712,13
2,Luna16 Lung Cancer Dataset,8.83,7.73,648.89 MB,1794,10
3,NIH Chest X ray 14,839.28,422.42,41.98 GB,112128,24
4,Pancreas Cancer,588.49,425.66,45.95 GB,3,0



**************************************************
FORMATOS POR DATASET
**************************************************

ISIC 2019:
  - .jpg: 25,331
  - .csv: 2
  - .txt: 2

Knee Osteoarthritis Classification:
  - .jpg: 8,395
  - .png: 4,317

Luna16 Lung Cancer Dataset:
  - .mhd: 888
  - .zraw: 888
  - .csv: 10
  - .py: 4
  - .txt: 3
  - .png: 1

NIH Chest X ray 14:
  - .png: 112,120
  - .pdf: 4
  - .csv: 2
  - .txt: 2

Pancreas Cancer:
  - sin extension: 1
  - .zip: 1
  - .md: 1


---
## Conversion a MHA para Entrenamiento del Router

Con base en el analisis anterior sabemos exactamente como esta construido cada dataset.
Las siguientes celdas convierten cada uno al formato `.mha` y los guardan en
`PROYECTO_MOE_VISION/02_Data_Processed/`. Al terminar cada conversion se borra
la carpeta descomprimida local para liberar espacio en Colab.


In [ ]:
!pip install -q SimpleITK nibabel

import os, time, glob, shutil
import numpy as np
import pandas as pd
import SimpleITK as sitk

LOCAL_DEST = '/content/datasets/'
PROC_BASE  = '/content/drive/MyDrive/PROYECTO_MOE_VISION/02_Data_Processed/'
os.makedirs(PROC_BASE, exist_ok=True)

def format_size(size_bytes):
    for unit in ['B','KB','MB','GB','TB']:
        if size_bytes < 1024.0:
            return f'{size_bytes:.2f} {unit}'
        size_bytes /= 1024.0

print('Imports OK.')


### Dataset 1 — NIH Chest X-ray 14 (PNG -> MHA)

Estructura real:
```
NIH Chest X ray 14/
  images_001/images/*.png
  images_002/images/*.png
  ...
  images_012/images/*.png
  Data_Entry_2017.csv
```
Solucion: `glob('**/*.png', recursive=True)` encuentra todos los PNG
independientemente de cuantas subcarpetas haya.


In [ ]:
NIH_SRC  = os.path.join(LOCAL_DEST, 'NIH Chest X ray 14')
NIH_DEST = os.path.join(PROC_BASE,  'NIH_Chest')
os.makedirs(NIH_DEST, exist_ok=True)

# CSV maestro
nih_csv = os.path.join(NIH_SRC, 'Data_Entry_2017.csv')
df_nih  = pd.read_csv(nih_csv).set_index('Image Index')
print(f'CSV NIH: {len(df_nih)} registros.')

# Busqueda recursiva de todos los PNG (incluyendo subcarpetas images/)
all_pngs = sorted(glob.glob(os.path.join(NIH_SRC, '**', '*.png'), recursive=True))
print(f'PNG encontrados (recursivo): {len(all_pngs)}')

converted, skipped, errors = 0, 0, 0
start = time.time()

for png_path in all_pngs:
    fname    = os.path.basename(png_path)
    out_path = os.path.join(NIH_DEST, fname.replace('.png', '.mha'))

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        img = sitk.ReadImage(png_path)

        if fname in df_nih.index:
            row = df_nih.loc[fname]
            img.SetMetaData('PatientID',     str(row['Patient ID']))
            img.SetMetaData('FindingLabels', str(row['Finding Labels']))
            img.SetMetaData('PatientGender', str(row['Patient Gender']))
            img.SetMetaData('ViewPosition',  str(row['View Position']))
        img.SetMetaData('RouterLabel', 'NIH_Chest')

        sitk.WriteImage(img, out_path, useCompression=True)
        converted += 1

        if converted % 5000 == 0:
            print(f'  Progreso: {converted}/{len(all_pngs)}...')

    except Exception as e:
        errors += 1

elapsed = time.time() - start
print(f'NIH Chest: {converted} convertidas | {skipped} saltadas | {errors} errores | {elapsed:.1f}s')

# Borrar carpeta fuente si la conversion fue exitosa
if errors == 0 and converted + skipped == len(all_pngs):
    shutil.rmtree(NIH_SRC)
    print(f'Carpeta fuente eliminada: {NIH_SRC}')
else:
    print(f'No se elimino la carpeta fuente por errores. Revisar manualmente.')


### Dataset 2 — ISIC 2019 (JPG RGB -> MHA)

Estructura real:
```
ISIC 2019/
  ISIC_2019_Training_Input/
    ISIC_2019_Training_Input/
      *.jpg
  ISIC_2019_Training_GroundTruth.csv
  ISIC_2019_Training_Metadata.csv
```


In [ ]:
ISIC_BASE  = os.path.join(LOCAL_DEST, 'ISIC 2019')
ISIC_DEST  = os.path.join(PROC_BASE, 'ISIC_2019')
os.makedirs(ISIC_DEST, exist_ok=True)

gt   = pd.read_csv(os.path.join(ISIC_BASE, 'ISIC_2019_Training_GroundTruth.csv'))
meta = pd.read_csv(os.path.join(ISIC_BASE, 'ISIC_2019_Training_Metadata.csv'))
df_isic = pd.merge(gt, meta, on='image').set_index('image')
CLASSES = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC','UNK']
print(f'ISIC CSV: {len(df_isic)} registros.')

# Busqueda recursiva de JPGs
all_jpgs = sorted(glob.glob(os.path.join(ISIC_BASE, '**', '*.jpg'), recursive=True))
print(f'JPG encontrados (recursivo): {len(all_jpgs)}')

converted, skipped, errors = 0, 0, 0
start = time.time()

for jpg_path in all_jpgs:
    img_id   = os.path.splitext(os.path.basename(jpg_path))[0]
    out_path = os.path.join(ISIC_DEST, f'{img_id}.mha')

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        img = sitk.ReadImage(jpg_path)

        if img_id in df_isic.index:
            row = df_isic.loc[img_id]
            diagnosis = next((c for c in CLASSES if row.get(c, 0) == 1.0), 'Unknown')
            img.SetMetaData('Diagnosis',    diagnosis)
            img.SetMetaData('PatientAge',   str(row.get('age_approx', '')))
            img.SetMetaData('AnatomicSite', str(row.get('anatom_site_general', '')))
            img.SetMetaData('Sex',          str(row.get('sex', '')))
        img.SetMetaData('RouterLabel', 'ISIC_2019')

        sitk.WriteImage(img, out_path, useCompression=True)
        converted += 1

        if converted % 5000 == 0:
            print(f'  Progreso: {converted}/{len(all_jpgs)}...')

    except Exception as e:
        errors += 1

elapsed = time.time() - start
print(f'ISIC 2019: {converted} convertidas | {skipped} saltadas | {errors} errores | {elapsed:.1f}s')

if errors == 0 and converted + skipped == len(all_jpgs):
    shutil.rmtree(ISIC_BASE)
    print(f'Carpeta fuente eliminada: {ISIC_BASE}')


### Dataset 3 — LUNA16 Pulmones 3D (MHD+ZRAW -> MHA)


In [ ]:
LUNA_BASE = os.path.join(LOCAL_DEST, 'Luna16 Lung Cancer Dataset')
LUNA_DEST = os.path.join(PROC_BASE, 'LUNA16')
os.makedirs(LUNA_DEST, exist_ok=True)

ann_csv = os.path.join(LUNA_BASE, 'annotations.csv')
df_ann  = pd.read_csv(ann_csv)
nodules_per_scan = df_ann.groupby('seriesuid').size().to_dict()

# Busqueda recursiva de MHDs
mhd_files = sorted(glob.glob(os.path.join(LUNA_BASE, '**', '*.mhd'), recursive=True))
print(f'MHD encontrados (recursivo): {len(mhd_files)}')

converted, skipped, errors = 0, 0, 0
start = time.time()

for mhd_path in mhd_files:
    series_uid = os.path.splitext(os.path.basename(mhd_path))[0]
    out_path   = os.path.join(LUNA_DEST, f'{series_uid}.mha')

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        ct_vol = sitk.ReadImage(mhd_path)
        ct_vol.SetMetaData('SeriesUID',   series_uid)
        ct_vol.SetMetaData('NoduleCount', str(nodules_per_scan.get(series_uid, 0)))
        ct_vol.SetMetaData('RouterLabel', 'LUNA16')

        sitk.WriteImage(ct_vol, out_path, useCompression=True)
        converted += 1

    except Exception as e:
        errors += 1
        print(f'Error en {series_uid}: {e}')

elapsed = time.time() - start
print(f'LUNA16: {converted} convertidos | {skipped} saltados | {errors} errores | {elapsed:.1f}s')

if errors == 0 and converted + skipped == len(mhd_files):
    shutil.rmtree(LUNA_BASE)
    print(f'Carpeta fuente eliminada: {LUNA_BASE}')


### Dataset 4 — Pancreas Cancer 3D (NIfTI -> MHA)

El dataset tiene un ZIP interno. Se descomprime recursivamente, luego
se buscan todos los `.nii.gz` y se convierten aplicando ventana HU abdominal.


In [ ]:
PANC_BASE = os.path.join(LOCAL_DEST, 'Pancreas Cancer')
PANC_DEST = os.path.join(PROC_BASE,  'Pancreas')
os.makedirs(PANC_DEST, exist_ok=True)

# Descomprimir ZIP interno si existe
inner_zips = sorted(glob.glob(os.path.join(PANC_BASE, '**', '*.zip'), recursive=True))
for iz in inner_zips:
    print(f'Descomprimiendo ZIP interno: {iz}')
    out_inner = os.path.dirname(iz)
    !unzip -q "{iz}" -d "{out_inner}"
    os.remove(iz)
    print('ZIP interno eliminado.')

# Busqueda recursiva de NIfTI
nii_files = sorted(glob.glob(os.path.join(PANC_BASE, '**', '*.nii.gz'), recursive=True))
nii_files += sorted(glob.glob(os.path.join(PANC_BASE, '**', '*.nii'), recursive=True))
print(f'NIfTI encontrados: {len(nii_files)}')

HU_MIN, HU_MAX = -200, 400

converted, skipped, errors = 0, 0, 0
start = time.time()

for nii_path in nii_files:
    basename = os.path.basename(nii_path).replace('.nii.gz', '').replace('.nii', '')
    out_path = os.path.join(PANC_DEST, f'{basename}.mha')

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        vol = sitk.ReadImage(nii_path)
        arr = sitk.GetArrayFromImage(vol).astype(np.float32)
        arr = np.clip(arr, HU_MIN, HU_MAX)
        arr = (arr - HU_MIN) / (HU_MAX - HU_MIN)

        new_vol = sitk.GetImageFromArray(arr)
        new_vol.CopyInformation(vol)
        new_vol.SetMetaData('RouterLabel', 'Pancreas')
        new_vol.SetMetaData('HU_Window',   f'[{HU_MIN}, {HU_MAX}]')

        sitk.WriteImage(new_vol, out_path, useCompression=True)
        converted += 1

    except Exception as e:
        errors += 1
        print(f'Error en {basename}: {e}')

elapsed = time.time() - start
print(f'Pancreas: {converted} convertidos | {skipped} saltados | {errors} errores | {elapsed:.1f}s')

if errors == 0 and converted + skipped == len(nii_files):
    shutil.rmtree(PANC_BASE)
    print(f'Carpeta fuente eliminada: {PANC_BASE}')


### Verificacion Final


In [ ]:
print('ARCHIVOS MHA EN Drive')
print('='*50)
for name in ['NIH_Chest', 'ISIC_2019', 'LUNA16', 'Pancreas']:
    path = os.path.join(PROC_BASE, name)
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith('.mha')]
        total = sum(os.path.getsize(os.path.join(path,f)) for f in files)
        print(f'  {name}: {len(files):,} MHA | {format_size(total)}')
    else:
        print(f'  {name}: no encontrado')
print('='*50)
